# Automated Hyperparameter Optimization and Model Selection for NLP Pipelines

*NLPAICS 2026 Summer School · Ernesto Luis Estevanell*

In the talk we walked through an idea: instead of hand-tuning a model, we can let a search do the work for us, and with AutoGOAL we can search over whole pipelines, not just their settings. This notebook is where we actually do it.

We'll stay with one small text-classification problem the whole way through, the 20 Newsgroups corpus, and get a little more ambitious in three steps. First we tune a classifier by hand, the way you've probably done before. Then we hand the whole pipeline over to AutoGOAL and let it build one for us. Finally we teach AutoGOAL something new: we add our own building block, a pretrained transformer, and let the search decide whether to use it.

Run the cells in order. Now and then I'll stop and ask you to change something and run it again; those are the moments where it sticks.

## Getting set up

First let's check that the kernel has everything we need, and then load the data we'll use throughout.

In [ ]:
# --- 0.1  Sanity check: right kernel, AutoGOAL present -------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
# Run fully offline: the model + data are pre-cached by setup.sh, so we never let
# HuggingFace phone home mid-session (a blocking call that stalls on conference wifi).
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

if sys.version_info[:2] not in [(3, 9), (3, 10)]:
    raise SystemExit(f"Kernel is Python {sys.version.split()[0]}, AutoGOAL needs 3.9/3.10. "
                     "Fix: Kernel → Change Kernel → 'NLPAICS 07'.")
try:
    import autogoal
    from autogoal_contrib import find_classes
except ModuleNotFoundError as e:
    raise SystemExit(f"AutoGOAL not in this kernel (missing '{e.name}'). "
                     "Pick the 'NLPAICS 07' kernel and make sure ./setup.sh 07 finished.")

# AutoGOAL evaluates each candidate pipeline in a worker process. On Linux (this
# vast.ai image) that uses fork, which shares memory with the kernel, so notebook-
# defined classes and cached data are visible to the workers. We pin it explicitly
# (a no-op on Linux; makes the notebook behave the same if you run it on macOS).
import multiprocessing as mp
try:
    mp.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass

from collections import Counter
algos = find_classes()
print("Python", sys.version.split()[0])
print(f"AutoGOAL ready, {len(algos)} building blocks discovered:",
      dict(Counter(c.__module__.split('.')[0] for c in algos)))

In [ ]:
# --- 0.2  The data: 20 Newsgroups, four topics --------------------------------
from sklearn.datasets import fetch_20newsgroups

CATEGORIES = ['sci.med', 'sci.space', 'rec.autos', 'talk.politics.misc']
# remove headers/footers/quotes so the model learns *language*, not giveaway metadata
strip = ('headers', 'footers', 'quotes')
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)

print(f"train: {len(train.data)} docs   test: {len(test.data)} docs   classes: {train.target_names}")
print("class balance (train):", dict(Counter(train.target_names[t] for t in train.target)))
print("\n--- an example document (first 280 chars) ---\n")
print(train.data[0][:280], "...")
print("\nlabel:", train.target_names[train.target[0]])

### How we'll use the data

20 Newsgroups already comes divided into a training set and a test set, so there is nothing for us to split by hand. What matters is how we use each one, and since the rule stays the same for the whole notebook, I'll say it once, here.

The test set is the final exam. We do not look at it while we work; we open it once, at the end of each part, only to report an honest number.

Everything else, every model and every setting we compare, is judged on the training set. And we don't even have to carve out a validation set ourselves. In Part 1, scikit-learn's cross-validation does it: it splits the training set into folds and rotates through them. In Parts 2 and 3, AutoGOAL does its own internal validation: it holds back a slice of the training data to score each pipeline it tries. So the only split we keep track of is the simple one, training data for choosing, test set for the final check.

## Part 1: Tuning a classifier by hand

Let's start somewhere familiar. A text classifier is really two pieces stuck together: something that turns text into numbers (a vectorizer), and something that learns from those numbers (a classifier). Each piece has settings, and choosing good ones is what we mean by hyperparameter tuning. We'll begin with the plainest version that could work, and then try to improve it.

In [ ]:
# --- 1.1  The baseline: simplest reasonable pipeline, no tuning ----------------
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC

baseline = Pipeline([('vec', CountVectorizer()),
                     ('clf', LinearSVC(max_iter=5000))]).fit(train.data, train.target)

print(f"Baseline (CountVectorizer → LinearSVC), test accuracy = "
      f"{baseline.score(test.data, test.target):.1%}")

So our starting point is 72.1%, from nothing more than word counts and an off-the-shelf linear SVM.

(If you have seen 20 Newsgroups score around 90% elsewhere, that was almost certainly with the headers and signatures left in, which more or less give the answer away. We stripped those out, so this number reflects the model actually reading the text.)

Now, how much can we gain just by tuning? Even this little two-step pipeline has plenty of knobs: how many words to keep, whether to use single words or pairs, whether to drop common stop-words, how to weight the counts, and the classifier's own regularization. Try a few values of each and the combinations pile up quickly.

In [ ]:
# --- 1.2  Count the search space ----------------------------------------------
from math import prod
knobs = {
    'ngram_range':  [(1,1), (1,2), (1,3)],
    'min_df':       [1, 2, 5],
    'max_df':       [0.5, 0.75, 1.0],
    'stop_words':   [None, 'english'],
    'sublinear_tf': [False, True],
    'C':            [0.03, 0.1, 0.3, 1, 3, 10],
}
print("configurations in a full grid:", prod(len(v) for v in knobs.values()))
print("…and that is before we even consider a different classifier.")

That is 648 combinations, and this is still a small grid. Grid search simply tries every one of them and keeps whichever does best under cross-validation. Let's run a slice of it.

In [ ]:
# --- 1.3  Grid search: try every combination, pick the best by CV -------------
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([('vec', TfidfVectorizer()), ('clf', LinearSVC(max_iter=5000))])
grid = {'vec__ngram_range':  [(1,1), (1,2)],
        'vec__min_df':       [1, 2, 5],
        'vec__stop_words':   [None, 'english'],
        'vec__sublinear_tf': [False, True],
        'clf__C':            [0.1, 0.3, 1, 3, 10]}

gs = GridSearchCV(pipe, grid, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"grid: {len(gs.cv_results_['params'])} configs × 5-fold CV")
print(f"best cross-val accuracy = {gs.best_score_:.1%}")
print(f"TEST accuracy           = {gs.score(test.data, test.target):.1%}")
print("winning knobs:", gs.best_params_)

We went from 72.1% to 84.1% on the test set, just by switching to TF-IDF, allowing word pairs, dropping English stop-words, and tuning the regularization. Notice that cross-validation looked a little better than the test score; it usually does.

The catch with grids is that they grow fast. This small one was already 648 points, and one more knob would push it into the thousands. Random search gets around that by sampling combinations instead of listing them all. Bergstra and Bengio showed back in 2012 that this often finds a good model sooner, because only a handful of knobs really matter, and random sampling does not waste time on the rest.

In [ ]:
# --- 1.4  Random search: sample the space instead of enumerating it -----------
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

space = {'vec__ngram_range':  [(1,1), (1,2), (1,3)],
         'vec__min_df':       [1, 2, 3, 5],
         'vec__max_df':       [0.5, 0.75, 1.0],
         'vec__stop_words':   [None, 'english'],
         'vec__sublinear_tf': [False, True],
         'clf__C':            loguniform(1e-2, 1e2)}      # a *distribution*, not a list

rs = RandomizedSearchCV(pipe, space, n_iter=40, cv=5, random_state=0, n_jobs=-1)
rs.fit(train.data, train.target)
print(f"random: 40 sampled configs × 5-fold CV")
print(f"best cross-val accuracy = {rs.best_score_:.1%}")
print(f"TEST accuracy           = {rs.score(test.data, test.target):.1%}")
print("winning knobs:", {k: (round(v,3) if isinstance(v,float) else v) for k,v in rs.best_params_.items()})

Forty random tries got us to 83.8%, matching the grid for a fraction of the work. That is the case for searching smartly rather than exhaustively.

But step back and notice how much we still decided ourselves. We fixed the shape of the pipeline, always a vectorizer and then a classifier. We picked which knobs to tune and what ranges to try. And we never once questioned reaching for a linear SVM. What if another classifier would do better here? What if an extra step would help? Searching over whole pipelines, and not just their settings, is what AutoML does, and that is where we are going next.

**Your turn.** Right now the classifier is fixed. Let's make the choice of classifier part of the search as well. In the cell below, compare the linear SVM against logistic regression and multinomial naive Bayes. (One catch: naive Bayes needs non-negative features, so keep TF-IDF.) Which one comes out ahead here?

In [ ]:
# 🟢 YOUR TURN 1, make the classifier a choice.
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

pipe2 = Pipeline([('vec', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
                  ('clf', LinearSVC(max_iter=5000))])

grid2 = {
    # TODO: add 'clf' : [LinearSVC(max_iter=5000), LogisticRegression(max_iter=2000), MultinomialNB()]
    # TODO: add a knob or two, e.g. 'vec__sublinear_tf': [False, True]
}

# gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
# print(gs2.best_score_, gs2.best_params_)

Give it a try first. When you are ready, here is one way to do it: `GridSearchCV` is happy to treat the whole classifier as just another thing to vary.

In [ ]:
# Solution to Your turn 1
grid2 = {
    'clf': [LinearSVC(max_iter=5000), LogisticRegression(max_iter=2000), MultinomialNB()],
    'vec__sublinear_tf': [False, True],
}
gs2 = GridSearchCV(pipe2, grid2, cv=5, n_jobs=-1).fit(train.data, train.target)
print(f"best CV = {gs2.best_score_:.1%}  with  {type(gs2.best_params_['clf']).__name__}")
print("full ranking:")
import numpy as np
for acc, p in sorted(zip(gs2.cv_results_['mean_test_score'], gs2.cv_results_['params']), reverse=True):
    print(f"  {acc:.1%}  {type(p['clf']).__name__:20s} sublinear_tf={p['vec__sublinear_tf']}")

In [ ]:
# --- 💾 CHECKPOINT 1, the best hand-tuned pipeline, self-contained ------------
# (Run this if you arrived late or your kernel restarted; it rebuilds Part 1's result.)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
best_naive = Pipeline([('vec', TfidfVectorizer(stop_words='english', ngram_range=(1,2),
                                               sublinear_tf=False, min_df=1)),
                       ('clf', LinearSVC(C=1, max_iter=5000))]).fit(train.data, train.target)
print(f"Checkpoint 1, hand-tuned pipeline test accuracy = "
      f"{best_naive.score(test.data, test.target):.1%}   (baseline was {0.721:.1%})")

## Part 2: Handing the whole pipeline to AutoGOAL

So far we built the pipeline ourselves and only let the search adjust its settings. Now we turn that around. We hand AutoGOAL a box of building blocks and a time budget, and it assembles the pipeline for us, choosing the steps, their order, and all their settings.

As I mentioned in the talk, AutoGOAL does this with a probabilistic grammar. Picture every valid pipeline as a sentence the grammar can produce; the search gradually learns which sentences tend to score well. The building blocks come from scikit-learn and NLTK, and what keeps them from being glued together nonsensically is a type system that knows what can feed into what. Let's look at those pieces one at a time, and then run a search.

### The building blocks

The first thing AutoGOAL needs is its box of parts. `find_classes()` is the catalogue of everything it can use, and we can filter it to see what is there.

In [ ]:
# --- 2.1  What can AutoGOAL use? find_classes() is the catalogue --------------
from autogoal_contrib import find_classes

print("total building blocks:", len(find_classes()))

# Filter by where they come from …
import autogoal_sklearn, autogoal_nltk
print("  from scikit-learn:", len(find_classes(modules=[autogoal_sklearn])))
print("  from NLTK        :", len(find_classes(modules=[autogoal_nltk])))

# … or by their NAME (a regex, matched against the class):
vectorizers = find_classes(include="Vectorizer")
print("\nvectorizers (name matches 'Vectorizer'):", sorted({c.__name__ for c in vectorizers}))

# … or by what they DO, keep blocks whose output is a class-label vector. (We compare
# the type object itself: the alias `VectorCategorical` prints as `Tensor[1,Categorical,
# Dense]`, so a name regex like output="VectorCategorical" would miss, match the type.)
from autogoal.kb import VectorCategorical
classifiers = [c for c in find_classes() if c.output_type() == VectorCategorical]
print(f"\nclassifiers (output is a class label): {len(classifiers)}")
print("  ", sorted({c.__name__ for c in classifiers})[:12], "…")

### How the pieces know how to connect

Here is the question that makes pipeline search possible at all: how does AutoGOAL know a vectorizer can feed a classifier, but not the other way around? Every block declares what it takes in and what it gives back, using what AutoGOAL calls semantic types, and one block can only follow another when the types line up.

The types are organised in a small hierarchy. A `Word` is a kind of `Sentence`, which is a kind of `Document`, which is a kind of `Text`. `Seq[X]` just means a sequence of `X` (our corpus is a `Seq[Sentence]`, a list of texts). The matrices that vectorizers produce have their own types, and `VectorCategorical` is a vector of class labels, which is what we want out at the end. The labels we feed in for training are wrapped as `Supervised[...]`, which tells the search they are the answers, handed in once, and not something to transform.

So our whole task fits on one line: we put in sentences and their labels, and we want predicted labels back. In AutoGOAL's terms, `(Seq[Sentence], Supervised[VectorCategorical]) → VectorCategorical`. We can even check these types ourselves.

In [ ]:
# --- 2.2  Types are checkable. Play with them. --------------------------------
from autogoal.kb import Sentence, Word, Document, Seq

print("isinstance('hello world', Sentence) :", isinstance('hello world', Sentence))
print("issubclass(Word, Sentence)          :", issubclass(Word, Sentence))   # a Word is a Sentence
print("issubclass(Seq[Word], Seq[Sentence]):", issubclass(Seq[Word], Seq[Sentence]))  # covariant

# Each block advertises its wiring via its run() annotations:
cv  = next(c for c in find_classes() if c.__name__ == 'CountVectorizer')
svc = next(c for c in find_classes() if c.__name__ == 'LinearSVC')
print("\nCountVectorizer :", cv.input_types(),  "->", cv.output_type())
print("LinearSVC       :", svc.input_types(), "->", svc.output_type())

### A peek under the hood (optional)

If you are curious how a search space gets described in the first place, here is a quick look. You can skip this and still follow everything that comes after; it is here for the curious.

In AutoGOAL you describe the settings of a block by annotating its `__init__`, and a small function turns that into a grammar you can sample from. It is the same idea as the random search from Part 1, but now it is a proper object that the pipeline search is built out of.

In [ ]:
# --- 2.3  Define a space, read it, sample from it -----------------------------
from autogoal.grammar import (generate_cfg, Sampler, Union,
                              ContinuousValue, CategoricalValue, BooleanValue, DiscreteValue)
from autogoal.utils import nice_repr

@nice_repr
class SVM:
    def __init__(self, C: ContinuousValue(0.1, 10),
                 kernel: CategoricalValue("linear", "rbf"),
                 shrinking: BooleanValue()):
        self.C, self.kernel, self.shrinking = C, kernel, shrinking

@nice_repr
class DTree:
    def __init__(self, max_depth: DiscreteValue(1, 10)):
        self.max_depth = max_depth

grammar = generate_cfg(Union("Classifier", SVM, DTree))   # "a Classifier is an SVM OR a DTree"
print(grammar, "\n")

s = Sampler(random_state=2)
print("four random configurations:")
for _ in range(4):
    print("   ", grammar.sample(sampler=s))

### The space of pipelines is a graph

Now scale that idea up from a single estimator to a whole pipeline. Given our input and output types and a box of blocks, AutoGOAL can lay out every pipeline that type-checks as a graph: each block is a node, an edge means two blocks fit together, and each node carries its own little space of settings. The nice part is that we can look at this graph, and even draw sample pipelines from it, before running any search at all.

In [ ]:
# --- 2.4  Build and inspect the pipeline graph (no search yet) ----------------
from autogoal.kb import build_pipeline_graph, Supervised, Sentence, VectorCategorical, Seq

# One curated box of blocks, vectorizers + linear text classifiers. We reuse this exact
# list for the live search in 2.5, so the graph you see IS the space that gets searched.
registry = [c for c in find_classes()
            if c.__name__ in {'CountVectorizer', 'TfidfVectorizer', 'LinearSVC', 'SGDClassifier',
                              'MultinomialNB', 'ComplementNB', 'LogisticRegression'}]
space = build_pipeline_graph(
    input_types=(Seq[Sentence], Supervised[VectorCategorical]),
    output_type=VectorCategorical,
    registry=registry,
)
print("blocks in this space:", sorted(a.__name__ for a in space.nodes()))
print("graph:", space.graph.number_of_nodes(), "nodes,", space.graph.number_of_edges(), "edges\n")
print("three pipelines sampled uniformly from the space:")
samp = Sampler(random_state=0)
for _ in range(3):
    print(space.sample(sampler=samp), "\n")

### Letting it search

Now we run the search for real. AutoGOAL walks that graph with an evolutionary search it calls PESearch: it tries pipelines, scores each one on a slice of the training data it holds out for validation, and gradually leans toward the kinds of pipelines that are working. We give it our types, the same box of blocks we just inspected, and a budget. I am scoring on a single train/validation split here to keep things quick. Watch the log as it runs; the best score should climb.

In [ ]:
# --- 2.5  Run the search --------------------------------------------------------
from autogoal.ml import AutoML
from autogoal.kb import Seq, Sentence, Supervised, VectorCategorical
from autogoal.search import RichLogger

# AutoGOAL wants string class labels (VectorCategorical); map ints -> topic names
y_train = [train.target_names[i] for i in train.target]
y_test  = [train.target_names[i] for i in test.target]

automl = AutoML(
    input=(Seq[Sentence], Supervised[VectorCategorical]),
    output=VectorCategorical,
    registry=registry,           # ← the same curated box we inspected in 2.4
    search_iterations=50,        # generations (the timeout usually binds first)
    pop_size=10,                 # pipelines per generation
    evaluation_timeout=20,       # seconds per pipeline
    search_timeout=90,           # total budget, the anytime "stop" knob
    cross_validation_steps=1,
    errors="ignore",
    random_state=0,
)
automl.fit(train.data, y_train, logger=RichLogger())

print("\nbest validation score:", automl.best_scores_)
print("best pipeline AutoGOAL built:\n", automl.best_pipelines_[0])

In [ ]:
# --- 2.5b  Open the test set, once --------------------------------------------
import numpy as np
pred = list(automl.predict(test.data))
acc = np.mean([p == t for p, t in zip(pred, y_test)])
print(f"AutoGOAL best pipeline, TEST accuracy = {acc:.1%}")

AutoGOAL built a TF-IDF and linear-classifier pipeline and reached 86.4% on the test set on our run. That is right in the same range as the pipeline we tuned by hand, except this time it chose the vectorizer, the classifier, and every setting itself, out of that box of blocks.

A couple of things to read off the output. `best_scores_` is a list, one entry per pipeline AutoGOAL kept, and the number is its score on AutoGOAL's own internal validation, so it sits a little above the test score. And the search is stochastic: give it more time and it tends to do better, run it again and it may settle on a slightly different pipeline. Your numbers will not match mine exactly, and that is expected.

### Two dials worth knowing

Two things you will reach for whenever you use this.

The first is the box of blocks itself, because that box is your search space. We handed AutoGOAL a short, curated list. The full `find_classes()` includes everything installed, and some of it (NLTK's sequence-tagging tools, for instance) makes no sense for classifying documents, so on a tight budget the search can waste time on it. Trimming the box to fit the task makes the search faster and usually better.

The second is the metric. By default AutoGOAL optimizes accuracy. If you cared about something else, say macro-F1 on an imbalanced problem, you would pass it as a one-line function. Our four classes are balanced, so it would not change much here, but it is worth knowing where the dial is:

```python
from sklearn.metrics import f1_score
AutoML(..., objectives=lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro"))
```

**Your turn** (this one runs a real search, so do it if you are ahead, or come back to it later). Try changing the search and see what moves: shrink the box to just TF-IDF and naive Bayes, or give it more time, or switch the metric. Then compare the scores.

In [ ]:
# 🟢 YOUR TURN 2, your own search. Edit the three marked lines.
my_registry = [c for c in find_classes()
               if c.__name__ in {'TfidfVectorizer', 'CountVectorizer', 'MultinomialNB',
                                  'ComplementNB', 'LinearSVC'}]   # ← (1) edit the block set
my = AutoML(
    input=(Seq[Sentence], Supervised[VectorCategorical]),
    output=VectorCategorical,
    registry=my_registry,
    search_iterations=30, pop_size=10,
    evaluation_timeout=20,
    search_timeout=60,        # ← (2) try 180
    cross_validation_steps=1, errors="ignore", random_state=0,
)
my.fit(train.data, y_train)
print(my.best_scores_, "\n", my.best_pipelines_[0])

In [ ]:
# --- 💾 CHECKPOINT 2, restart point for Part 3 (instant) ----------------------
# Jumping in here (or recovered from a restart)? Part 3 only needs the data + AutoGOAL.
# This reloads both so the rest runs standalone, Part 3 defines its own building blocks.
from sklearn.datasets import fetch_20newsgroups
from autogoal.ml import AutoML
from autogoal.kb import Seq, Sentence, Supervised, VectorCategorical
from autogoal_contrib import find_classes
CATEGORIES = ['sci.med', 'sci.space', 'rec.autos', 'talk.politics.misc']
strip = ('headers', 'footers', 'quotes')
train = fetch_20newsgroups(subset='train', categories=CATEGORIES, remove=strip, random_state=0)
test  = fetch_20newsgroups(subset='test',  categories=CATEGORIES, remove=strip, random_state=0)
y_train = [train.target_names[i] for i in train.target]
y_test  = [train.target_names[i] for i in test.target]
print(f"Checkpoint 2 ready, {len(train.data)} train / {len(test.data)} test docs loaded.")

## Part 3: Teaching AutoGOAL something new

This is the part I enjoy most. That box of blocks is not fixed, it is just a list of classes, so you can add your own.

A block is a small class that says two things: what it takes in and gives back (through the types on its `run` method), and what settings it has (through the annotations on its `__init__`). That is the whole contract, no registration step, no plugin system. Let's write a tiny one to see the shape of it, and then wrap a real pretrained transformer as a block and see whether the search wants it.

### A first, tiny block

The smallest thing that counts as a block: take a sentence, hand back a list of words. We subclass `AlgorithmBase`, put types on `run`, and annotate `__init__` with the settings we would like searched. AutoGOAL works out the rest by looking at it.

In [ ]:
# --- 3.1  The smallest possible custom block: a typed transform ----------------
from autogoal.kb import AlgorithmBase, Sentence, Seq, Word
from autogoal.grammar import BooleanValue, DiscreteValue
from autogoal.utils import nice_repr

@nice_repr
class LongWordTokenizer(AlgorithmBase):
    """Split a sentence into words, optionally lowercasing and dropping short tokens."""
    def __init__(self, min_length: DiscreteValue(0, 5), lower: BooleanValue()):
        self.min_length = min_length      # ← searchable: an integer in [0, 5]
        self.lower = lower                # ← searchable: True / False

    def run(self, input: Sentence) -> Seq[Word]:     # ← Sentence in, Seq[Word] out
        if self.lower:
            input = input.lower()
        return [w for w in input.split() if len(w) > self.min_length]

print("declared contract:", LongWordTokenizer.input_types(), "->", LongWordTokenizer.output_type())
print("example:", LongWordTokenizer(min_length=3, lower=True).run("The Apollo program reached the Moon"))

### Wrapping a transformer

Now the interesting one. We take a small pretrained sentence model, MiniLM, and wrap it as a block that turns text into dense vectors. Its `run` takes sentences and returns a matrix, exactly the shape a classifier expects, so once it is in the box the search can pick it or leave it, just like any other block.

(A quick note on words: MiniLM is an encoder, not a generative model like the LLMs you usually hear about. But the move is exactly the same if you wanted to drop in a big generative model as a feature extractor or a judge.)

In [ ]:
# --- 3.2  Wrap a transformer LM as an AutoGOAL block --------------------------
import numpy as np
from autogoal.kb import AlgorithmBase, Seq, Sentence, MatrixContinuousDense
from autogoal.grammar import BooleanValue
from autogoal.utils import nice_repr

@nice_repr
class TransformerEmbedding(AlgorithmBase):
    """Pretrained MiniLM sentence embeddings as features: Seq[Sentence] -> (n × d) dense
    matrix (d = the model's embedding size: 384 for MiniLM, 768 for mpnet)."""
    _MODEL = None
    _NAME  = "all-MiniLM-L6-v2"
    _VEC   = {}   # cache ONE raw vector per unique document, so any subset is a cache hit

    def __init__(self, normalize: BooleanValue()):
        self.normalize = normalize

    @classmethod
    def _model(cls):
        if cls._MODEL is None:
            import torch
            from sentence_transformers import SentenceTransformer
            device = "cuda" if torch.cuda.is_available() else "cpu"   # use the vast.ai GPU if present
            cls._MODEL = SentenceTransformer(cls._NAME, device=device)
        return cls._MODEL

    def run(self, input: Seq[Sentence]) -> MatrixContinuousDense:
        # encode only documents we haven't seen; warming the full corpus once (in 3.2b) means
        # the forked search workers inherit a full cache and never re-load the model.
        missing = [d for d in input if d not in TransformerEmbedding._VEC]
        if missing:
            V = self._model().encode(missing, batch_size=64, show_progress_bar=False,
                                     convert_to_numpy=True).astype("float32")
            for d, v in zip(missing, V):
                TransformerEmbedding._VEC[d] = v
        raw = np.vstack([TransformerEmbedding._VEC[d] for d in input])
        if self.normalize:
            n = np.linalg.norm(raw, axis=1, keepdims=True); n[n == 0] = 1.0
            return raw / n
        return raw

print("contract:", TransformerEmbedding.input_types(), "->", TransformerEmbedding.output_type())

In [ ]:
# --- 3.2b  Representation showdown, a learning curve (a *fair* comparison) ----
# Fair test: the SAME classifier (LogisticRegression) on both representations, scored on
# a held-out validation split, averaged over 3 random subsamples. Only the FEATURES differ,
# and we vary the amount of labelled data. (Encodes the whole pool once; quick on a GPU, a little slower on CPU.)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

y_all   = [train.target_names[i] for i in train.target]
EMB_ALL = TransformerEmbedding(normalize=True).run(list(train.data))   # encode all docs once

def val_acc(n, seed):
    idx = train_test_split(np.arange(len(train.data)), train_size=n,
                           random_state=seed, stratify=y_all)[0]
    y = np.array([y_all[i] for i in idx])
    tr, va = train_test_split(np.arange(n), test_size=0.25, random_state=seed, stratify=y)
    emb = LogisticRegression(max_iter=2000).fit(EMB_ALL[idx[tr]], y[tr]).score(EMB_ALL[idx[va]], y[va])
    docs = [train.data[i] for i in idx]
    tf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), sublinear_tf=True)
    Xtr = tf.fit_transform([docs[j] for j in tr]); Xva = tf.transform([docs[j] for j in va])
    tfidf = LogisticRegression(max_iter=2000).fit(Xtr, y[tr]).score(Xva, y[va])
    return emb, tfidf

print(f"{'train docs':>10} | {'MiniLM-emb':>11} | {'TF-IDF':>9} | winner")
print("-" * 48)
for n in (200, 400, 800, 1500, 2200):
    es, ts = zip(*[val_acc(n, s) for s in (0, 1, 2)])     # average over 3 subsamples
    e, t = np.mean(es), np.mean(ts)
    print(f"{n:>10} | {e:>10.1%} | {t:>8.1%} | {'MiniLM' if e > t else 'TF-IDF'}")

Look at where the two lines cross.

With only 200 labelled documents the transformer is far ahead, around 89% against 67%. It shows up already knowing the language. As we give both more data, TF-IDF, which starts from nothing, catches up, and by 2,200 documents it has actually pulled in front.

That is the whole intuition behind transfer learning in one picture: when labelled data is scarce, a pretrained model wins; when there is plenty of it, simple features can catch up and overtake. And it turns the vague question, "should I use a big model here?", into something concrete: it depends on how much labelled data you have.

(To keep this fair I used the same classifier on both sides and scored on a held-out split, averaged over a few subsamples, so what you are seeing is the features talking, not luck. The numbers still wobble a point or two from run to run.)

In [ ]:
# --- 3.3  Let AutoGOAL choose, with the transformer in the registry -----------
from autogoal.ml import AutoML
from autogoal.kb import Seq, Sentence, Supervised, VectorCategorical

# A low-data slice (800 docs), the regime where, above, the transformer was ahead.
N  = 800
Xs = list(train.data[:N])
ys = [train.target_names[i] for i in train.target][:N]

reg_llm = [TransformerEmbedding] + [c for c in find_classes()
          if c.__name__ in {'TfidfVectorizer','CountVectorizer',
                            'LinearSVC','LogisticRegression','SGDClassifier'}]
print("registry now includes your block:", sorted(getattr(c,'__name__',str(c)) for c in reg_llm))

automl_llm = AutoML(
    input=(Seq[Sentence], Supervised[VectorCategorical]),
    output=VectorCategorical,
    registry=reg_llm,                 # ← classic blocks AND your transformer, side by side
    search_iterations=30, pop_size=8,
    evaluation_timeout=40, search_timeout=120,
    cross_validation_steps=1, errors="ignore", random_state=0,
)
automl_llm.fit(Xs, ys)
print("\nbest validation score:", automl_llm.best_scores_)
print("AutoGOAL picked:\n", automl_llm.best_pipelines_[0])

On this small slice the search went with our transformer; on my run it paired it with a linear classifier, which is exactly what the curve said should win at this size. (It is a stochastic search, so the exact pairing and score will move around for you.)

But the real point is what just happened. We added one small class, and AutoGOAL weighed a pretrained transformer against the classic approaches on equal terms and told us which was worth using here. That is the bridge from the classical AutoML in the talk to the LLM era: a new kind of representation, an encoder today or a generative model tomorrow, is just another block to drop in the box.

**Your turn.** A few things worth trying:

1. Swap the model: point `_NAME` at `"all-mpnet-base-v2"` (stronger, 768 dimensions) and re-run the curve. At home that is about a 420 MB download, so you would first unset the offline flags (`os.environ.pop("HF_HUB_OFFLINE", None)`) or pre-cache it in `setup.sh`.
2. Add a setting: give `TransformerEmbedding` a pooling choice (`CategoricalValue("mean", "max")`) and use it in `run`.
3. Write a little featurizer of your own, say counts of a few domain keywords, and add it to the box.

### One more thing, if you are curious: your own type

The types are not fixed either. You can carve out a new one by subclassing an existing type and tightening what it matches. Here is an `Email` type that only accepts strings with an `@` in them; a block that asked for `Seq[Email]` would then turn down anything else.

In [ ]:
# ★ A custom semantic type, in five lines
from autogoal.kb import Text

class Email(Text):                       # Email ⊂ Text, automatically
    @classmethod
    def _match(cls, x):
        return super()._match(x) and "@" in x

print("isinstance('a@b.com', Email):", isinstance('a@b.com', Email))
print("isinstance('hello',   Email):", isinstance('hello', Email))
print("issubclass(Email, Text)     :", issubclass(Email, Text))

## Where we landed

We climbed the same ladder we sketched in the talk, and at every rung the machine did more of the work. We started by hand-tuning a fixed pipeline. We let grid and random search choose its settings. We handed the whole pipeline to AutoGOAL and let it choose the blocks too. And then we widened the box with a block of our own and let the search decide whether a transformer was worth it.

The idea underneath all of it is simple: AutoML is search over typed pipelines. You stay in charge of four things, the blocks you offer, the budget you allow, the metric you care about, and where the search may look, and you hand off the tedious part. Adding an LLM does not change the game; it is just one more block in the box.

If you would like to keep going: give the search more time, hand it the full set of blocks, or point it at your own data. `AutoML.fit` takes any list of texts and labels. Thanks for following along.